In [ ]:
#!/usr/bin/env python3
"""
Project 13 -- Move 1 Step 3: zero-shot trans-network POC ("the Corgi killer").
Run on Kaggle (network + no wall-clock-per-call constraints; same reason as
notebooks/kaggle_1m_scbloodnl_de.py -- this needs to stream the same two large
SCT-normalized count matrices, which this sandbox's 45s-per-call limit and
~5-12MB/s effective throughput can't handle). Companion: STATE.md 2026-07-15e/f/g
(the reQTL World-B pipeline this POC follows on from).

WHY THIS SCRIPT EXISTS
-----------------------
Gate 2's reQTL analysis (src/21, guardrail-fixed in src/23) established World B:
1M-scBloodNL's stimulation-response genes show no more cis-genetic signature than
matched non-responders, on a test cleaned of baseline-eQTL contamination. Brian's
reframe: this isn't a dead end, it's the wiring diagram -- cis-sequence encodes
network TOPOLOGY, trans-environment dictates DYNAMICS. Move 1 proposes a
compositional cis+trans model. Before committing to that full build, this POC
tests the cheapest falsifiable version of the trans half of that claim: can a
LIGHTWEIGHT trans-regulatory model (target-gene response as a function of
transcription-factor expression shifts) generalize ZERO-SHOT to an unseen
stimulation condition, and does it beat a model that structurally cannot see
condition-specific information at all (the "cis-sequence baseline")?

DESIGN
------
Unit of analysis: per (cell type, condition) MEAN PSEUDOBULK profile (not
per-donor -- keeps this a lightweight POC, not a full donor-resolved model;
per-donor trans-modeling is a natural next step if this POC succeeds, not
this script's job). 4 cell types (same frozen Calderon mapping as prior Gate 2
work) x 7 conditions (UT + 3hCA/24hCA/3hMTB/24hMTB/3hPA/24hPA) = 28 buckets,
built by streaming the same two SCT-normalized matrices as
kaggle_1m_scbloodnl_de.py (reused verbatim: fetch/stream_aggregate/
col_bucket_array/process_chemistry). No per-donor pairing required this time
(mean-pseudobulk unit doesn't need it), so ALL donors present in each condition
contribute, not just the 104 UT+stim-paired ones -- a real, deliberate design
loosening from the earlier DE work, documented not hidden.

delta(gene, celltype, condition) = mean_expr(condition) - mean_expr(UT), for
each of the 6 non-UT conditions, per cell type. (SCT-normalized values used
directly, same convention as kaggle_1m_scbloodnl_de.py -- already
variance-stabilized, differencing directly is standard for this data type.)

TF/regulator predictors: CollecTRI (via the `decoupler` package, a curated,
signed TF-target regulon resource -- already named in this repo's own
STATE.md "Key data" list). TF_delta(condition) = the delta vector restricted
to TF genes themselves (their own expression shift), used as the condition's
"trans state" fingerprint. Falls back to a small hardcoded list of well-known
human immune TFs if the package/network is unavailable (flagged in the report,
not silently substituted).

TRANS-MODEL (leave-one-condition-out, 6 folds): for held-out condition c,
compute Pearson similarity between TF_delta[:,c] and TF_delta[:,c'] for each
of the 5 training conditions c'. Predict target_delta(g, c) as the
similarity-WEIGHTED average of target_delta(g, c') across training conditions
(weights = max(similarity,0), renormalized; uniform if all non-positive). This
is the specific, falsifiable trans-regulatory hypothesis: genes respond
similarly when the driving TF state is similar, regardless of which pathogen
caused it.

CIS-SEQUENCE BASELINE: predict target_delta(g, c) as the UNWEIGHTED (uniform)
average of target_delta(g, c') across the same 5 training conditions. This is
the principled cis-only baseline for a zero-shot-condition test: a cis-sequence
model structurally CANNOT see which stimulus is active (sequence doesn't change
with the environment), so the best it can do for ANY held-out condition is
predict a gene's typical/average response -- constant across conditions,
varying only by gene. The trans-model and cis-baseline are constructed to
differ in EXACTLY ONE respect (similarity-weighted vs uniform averaging over
the same training data), isolating "does condition-specific TF-state
information help" as cleanly as this lightweight design allows.

EVALUATION: per (celltype, held-out condition) fold, Pearson r between
predicted and true delta across genes (reported for the full valid-gene panel
AND restricted to genes with a detectable response, |true delta|>=0.25, in the
held-out condition -- the harder/more relevant subset). Wilcoxon signed-rank
test across all 24 folds (4 celltypes x 6 conditions) on (r_trans - r_cis).
24h Candida is highlighted explicitly per the task brief's example, alongside
the full leave-one-out sweep (more robust than a single fold).

Output
------
    /kaggle/working/scbloodnl_move1_delta_<group>.parquet   (per-celltype delta matrix, all genes x 6 conditions)
    /kaggle/working/scbloodnl_move1_poc_folds.csv            (per-fold r_trans, r_cis, n_genes)
    /kaggle/working/scbloodnl_move1_poc_report.json          (full readout + bifurcation-style summary)
Bring these back into data/processed/ to log the POC result in STATE.md /
gate2_substrate_report.json.

Usage
-----
    python kaggle_1m_scbloodnl_move1_poc.py
"""
from __future__ import annotations
import gzip
import json
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

IS_KAGGLE = Path("/kaggle").exists()
WORK = Path("/kaggle/working") if IS_KAGGLE else Path(__file__).resolve().parent
RAW = WORK / "raw_1m_scbloodnl"
RAW.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://molgenis26.gcc.rug.nl/downloads/1m-scbloodnl"

GROUP_MAP = {
    "B": "B",
    "mono 1": "monocyte", "mono 2": "monocyte", "mono 3": "monocyte", "mono 4": "monocyte",
    "memory CD8T": "CD8T", "naive CD8T": "CD8T",
    "naive CD4T": "naive_CD4T",
}
GROUPS = ["B", "monocyte", "CD8T", "naive_CD4T"]
CONDITIONS = ["UT", "3hCA", "24hCA", "3hMTB", "24hMTB", "3hPA", "24hPA"]
STIM_CONDITIONS = [c for c in CONDITIONS if c != "UT"]

CHUNK_ROWS = 5_000_000
RESPONSE_THRESHOLD = 0.25  # |true delta| cutoff for the "detectable response" gene subset

# Fallback TF list (used only if `decoupler`/CollecTRI can't be fetched) -- well-known
# human immune/blood transcription factors, hand-curated, NOT a substitute for CollecTRI's
# curated regulon resource, flagged explicitly in the report if this path is taken.
FALLBACK_TF_LIST = [
    "STAT1", "STAT2", "STAT3", "STAT4", "STAT5A", "STAT5B", "STAT6", "IRF1", "IRF3", "IRF4",
    "IRF7", "IRF8", "NFKB1", "NFKB2", "RELA", "RELB", "REL", "JUN", "JUNB", "JUND", "FOS",
    "FOSB", "FOSL1", "FOSL2", "ATF3", "ATF4", "NR4A1", "NR4A2", "NR4A3", "EGR1", "EGR2",
    "EGR3", "MYC", "MYB", "GATA1", "GATA2", "GATA3", "TBX21", "RORC", "FOXP3", "SPI1",
    "CEBPA", "CEBPB", "CEBPD", "IKZF1", "IKZF2", "IKZF3", "BCL6", "PRDM1", "BATF", "BATF3",
    "EOMES", "RUNX1", "RUNX3", "TCF7", "LEF1", "ID2", "ID3", "KLF2", "KLF4", "KLF6",
    "SOCS1", "SOCS3", "PPARG", "XBP1", "ETS1", "ETS2", "ELF1", "ELK1", "MAF", "MAFB",
]


def fetch(fname, dest=None):
    dest = dest or (RAW / fname)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[fetch] {fname} already present ({dest.stat().st_size/1e6:.1f}MB), skip")
        return dest
    url = f"{BASE_URL}/{fname}"
    print(f"[fetch] downloading {url} ...")
    t0 = time.time()
    urllib.request.urlretrieve(url, dest)
    print(f"[fetch] done {fname}: {dest.stat().st_size/1e6:.1f}MB in {time.time()-t0:.1f}s")
    return dest


def build_bucket_map():
    """Barcode -> (group, condition) bucket. No donor-pairing requirement this time --
    mean-pseudobulk-per-(celltype,condition) doesn't need it (a deliberate loosening
    from kaggle_1m_scbloodnl_de.py's paired-donor design, documented in the module
    docstring). Uses ALL donors present in each condition."""
    ct_path = fetch("1M_cell_types.tsv")
    asn_path = fetch("1M_assignments_conditions_expid.tsv")
    ct = pd.read_csv(ct_path, sep="\t")
    asn = pd.read_csv(asn_path, sep="\t")
    meta = ct.merge(asn, on="barcode", how="inner")

    meta["group"] = meta.cell_type.map(GROUP_MAP)
    meta = meta.dropna(subset=["group", "assignment", "timepoint"]).copy()
    meta = meta[meta.timepoint.isin(CONDITIONS)].copy()

    buckets = sorted((meta.group + "|" + meta.timepoint).unique())
    bucket_id = {b: i + 1 for i, b in enumerate(buckets)}  # 1-indexed, 0 = unwanted
    meta["bucket_str"] = meta.group + "|" + meta.timepoint
    meta["bucket_id"] = meta.bucket_str.map(bucket_id)

    bucket_meta = pd.DataFrame(
        [(bid, *b.split("|")) for b, bid in bucket_id.items()],
        columns=["bucket_id", "group", "condition"]
    )
    print(f"[bucket] {len(buckets)} buckets across {len(GROUPS)} groups x {len(CONDITIONS)} conditions")
    print(bucket_meta.groupby(["group", "condition"]).size().rename("n_cells_placeholder"))
    return meta.set_index("barcode")["bucket_id"], bucket_meta


def col_bucket_array(barcode_bucket, chem_barcodes_path):
    bcs = pd.read_csv(chem_barcodes_path, header=None)[0]
    arr = bcs.map(barcode_bucket).fillna(0).astype(np.int64).values
    print(f"[colbucket] {chem_barcodes_path.name}: {len(bcs)} cells, {int((arr>0).sum())} wanted")
    return arr


def stream_aggregate(mtx_path, col_bucket, n_genes, n_buckets):
    """Returns (sum_mat, count_mat): (n_genes, n_buckets) float64 arrays. Verbatim from
    kaggle_1m_scbloodnl_de.py (proven correct there against a real 2M-row slice test)."""
    sum_mat = np.zeros((n_genes, n_buckets + 1), dtype=np.float64)
    cnt_mat = np.zeros((n_genes, n_buckets + 1), dtype=np.int64)

    t0 = time.time()
    with gzip.open(mtx_path, "rt") as f:
        f.readline()
        dims = f.readline().split()
        decl_genes, decl_cells, decl_nnz = int(dims[0]), int(dims[1]), int(dims[2])
        print(f"[stream] {mtx_path.name}: declared {decl_genes} genes x {decl_cells} cells, {decl_nnz} nnz")

        reader = pd.read_csv(f, sep=" ", header=None, names=["gene", "cell", "val"],
                              dtype={"gene": np.int32, "cell": np.int32, "val": np.float64},
                              chunksize=CHUNK_ROWS)
        n_rows_seen = 0
        for chunk in reader:
            b = col_bucket[chunk["cell"].values - 1]
            keep = b > 0
            if keep.any():
                g = chunk["gene"].values[keep] - 1
                bb = b[keep]
                v = chunk["val"].values[keep]
                np.add.at(sum_mat, (g, bb), v)
                np.add.at(cnt_mat, (g, bb), 1)
            n_rows_seen += len(chunk)
            if n_rows_seen % (CHUNK_ROWS * 5) == 0:
                print(f"  [stream] {n_rows_seen}/{decl_nnz} rows ({time.time()-t0:.0f}s)")
    print(f"[stream] {mtx_path.name} done: {n_rows_seen} rows in {time.time()-t0:.0f}s")
    return sum_mat[:, 1:], cnt_mat[:, 1:]


def process_chemistry(chem, barcode_bucket, n_buckets):
    features_path = fetch(f"10x_{chem}_SCT_features.tsv.gz")
    barcodes_path = fetch(f"10x_{chem}_barcodes.tsv.gz")
    matrix_path = fetch(f"10x_{chem}_SCT_matrix.mtx.gz")

    genes = pd.read_csv(features_path, header=None)[0].values
    n_genes = len(genes)
    col_bucket = col_bucket_array(barcode_bucket, barcodes_path)

    sum_mat, cnt_mat = stream_aggregate(matrix_path, col_bucket, n_genes, n_buckets)
    return genes, sum_mat, cnt_mat


def get_tf_list(all_genes_set):
    """Try CollecTRI via `decoupler` (pip-installable, lightweight). Falls back to a
    hardcoded list if unavailable -- flagged in the returned source string, not hidden."""
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "decoupler"],
                        check=True, timeout=120)
        import decoupler as dc
        net = dc.get_collectri(organism="human", split_complexes=False)
        tf_genes = sorted(set(net["source"]) & all_genes_set)
        if len(tf_genes) >= 20:
            print(f"[TF list] CollecTRI: {len(tf_genes)} TFs present in our gene panel "
                  f"(of {net['source'].nunique()} total in CollecTRI)")
            return tf_genes, "CollecTRI (decoupler.get_collectri, human)"
        print(f"[TF list] CollecTRI returned only {len(tf_genes)} usable TFs -- falling back")
    except Exception as e:
        print(f"[TF list] CollecTRI fetch failed ({e}) -- falling back to hardcoded list")
    tf_genes = sorted(set(FALLBACK_TF_LIST) & all_genes_set)
    print(f"[TF list] FALLBACK hardcoded list: {len(tf_genes)} TFs present in our gene panel "
          f"(of {len(FALLBACK_TF_LIST)} total)")
    return tf_genes, "hardcoded fallback list (CollecTRI unavailable)"


def run_poc_for_group(group, delta_df, tf_genes):
    """delta_df: index=gene, columns=STIM_CONDITIONS, values=delta vs UT. Returns list of
    per-fold result dicts (leave-one-condition-out)."""
    tf_in_panel = [t for t in tf_genes if t in delta_df.index]
    valid_genes = delta_df.dropna(how="any").index  # genes with a delta in ALL 6 conditions
    tf_valid = [t for t in tf_in_panel if t in valid_genes]
    if len(tf_valid) < 5 or len(valid_genes) < 50:
        print(f"[{group}] insufficient valid genes/TFs (n_genes={len(valid_genes)}, "
              f"n_tf={len(tf_valid)}) -- skipped")
        return []

    D = delta_df.loc[valid_genes]  # (n_genes, 6)
    TFD = D.loc[tf_valid]          # (n_tf, 6)

    fold_results = []
    for held_out in STIM_CONDITIONS:
        train_conds = [c for c in STIM_CONDITIONS if c != held_out]
        tf_held = TFD[held_out].values
        sims = {}
        for c in train_conds:
            r, _ = stats.pearsonr(tf_held, TFD[c].values)
            sims[c] = r if np.isfinite(r) else 0.0
        pos_sims = {c: max(v, 0.0) for c, v in sims.items()}
        wsum = sum(pos_sims.values())
        weights_trans = {c: (v / wsum if wsum > 0 else 1.0 / len(train_conds)) for c, v in pos_sims.items()}
        weights_cis = {c: 1.0 / len(train_conds) for c in train_conds}  # uniform, condition-blind

        pred_trans = sum(weights_trans[c] * D[c].values for c in train_conds)
        pred_cis = sum(weights_cis[c] * D[c].values for c in train_conds)
        true_vals = D[held_out].values

        r_trans, _ = stats.pearsonr(pred_trans, true_vals)
        r_cis, _ = stats.pearsonr(pred_cis, true_vals)

        responsive = np.abs(true_vals) >= RESPONSE_THRESHOLD
        if responsive.sum() >= 20:
            r_trans_resp, _ = stats.pearsonr(pred_trans[responsive], true_vals[responsive])
            r_cis_resp, _ = stats.pearsonr(pred_cis[responsive], true_vals[responsive])
        else:
            r_trans_resp, r_cis_resp = np.nan, np.nan

        fold_results.append({
            "group": group, "held_out_condition": held_out, "n_genes": int(len(valid_genes)),
            "n_tf": int(len(tf_valid)), "n_responsive_genes": int(responsive.sum()),
            "r_trans_all_genes": float(r_trans), "r_cis_all_genes": float(r_cis),
            "r_trans_responsive_only": float(r_trans_resp), "r_cis_responsive_only": float(r_cis_resp),
            "trans_minus_cis_all": float(r_trans - r_cis),
            "trans_minus_cis_responsive": float(r_trans_resp - r_cis_resp) if np.isfinite(r_trans_resp) else None,
            "tf_similarity_to_training": sims,
        })
        print(f"[{group}] held_out={held_out}: r_trans={r_trans:.3f} r_cis={r_cis:.3f} "
              f"(responsive-only: r_trans={r_trans_resp:.3f} r_cis={r_cis_resp:.3f}, n={int(responsive.sum())})")

    return fold_results


def main():
    t0 = time.time()
    barcode_bucket, bucket_meta = build_bucket_map()
    n_buckets = bucket_meta.bucket_id.max()

    genes_v2, sum_v2, cnt_v2 = process_chemistry("v2", barcode_bucket, n_buckets)
    genes_v3, sum_v3, cnt_v3 = process_chemistry("v3", barcode_bucket, n_buckets)

    all_genes = sorted(set(genes_v2) | set(genes_v3))
    gidx = {g: i for i, g in enumerate(all_genes)}
    n_all = len(all_genes)
    sum_combined = np.zeros((n_all, n_buckets), dtype=np.float64)
    cnt_combined = np.zeros((n_all, n_buckets), dtype=np.int64)

    for genes, sum_mat, cnt_mat in [(genes_v2, sum_v2, cnt_v2), (genes_v3, sum_v3, cnt_v3)]:
        idx_map = np.array([gidx[g] for g in genes])
        np.add.at(sum_combined, idx_map, sum_mat)
        np.add.at(cnt_combined, idx_map, cnt_mat)

    print(f"[combine] {n_all} total genes (v2={len(genes_v2)}, v3={len(genes_v3)}) ({time.time()-t0:.0f}s)")
    mean_mat = np.divide(sum_combined, cnt_combined, out=np.full_like(sum_combined, np.nan), where=cnt_combined > 0)
    # bucket_id is 1-indexed and mean_mat's columns are 0-indexed by bucket_id-1 (stream_aggregate
    # drops the dummy col 0); build column labels from bucket_meta SORTED by bucket_id so column i
    # of mean_mat lines up with bucket_id i+1 -- not from itertuples() in whatever order bucket_meta
    # happens to be in, which would silently mislabel every column.
    col_order = bucket_meta.sort_values("bucket_id")
    mean_df = pd.DataFrame(mean_mat, index=all_genes,
                            columns=[f"{r.group}|{r.condition}" for r in col_order.itertuples()])

    tf_genes, tf_source = get_tf_list(set(all_genes))

    report = {"n_all_genes": n_all, "n_buckets": int(n_buckets), "tf_source": tf_source,
              "n_tf_total": len(tf_genes), "groups": {}}
    all_folds = []

    for group in GROUPS:
        cols_ut = f"{group}|UT"
        if cols_ut not in mean_df.columns:
            print(f"[{group}] no UT bucket -- skipped")
            continue
        delta = {}
        for cond in STIM_CONDITIONS:
            col = f"{group}|{cond}"
            if col in mean_df.columns:
                delta[cond] = mean_df[col] - mean_df[cols_ut]
        delta_df = pd.DataFrame(delta)
        delta_path = WORK / f"scbloodnl_move1_delta_{group}.parquet"
        delta_df.to_parquet(delta_path)
        print(f"[{group}] delta matrix {delta_df.shape} written to {delta_path} ({time.time()-t0:.0f}s)")

        fold_results = run_poc_for_group(group, delta_df, tf_genes)
        all_folds.extend(fold_results)
        report["groups"][group] = {
            "n_conditions_available": len([c for c in STIM_CONDITIONS if f"{group}|{c}" in mean_df.columns]),
            "n_folds_run": len(fold_results),
        }

    folds_df = pd.DataFrame([{k: v for k, v in f.items() if k != "tf_similarity_to_training"} for f in all_folds])
    folds_df.to_csv(WORK / "scbloodnl_move1_poc_folds.csv", index=False)

    # ---------- bifurcation-style pooled readout ----------
    valid_resp = folds_df.dropna(subset=["r_trans_responsive_only", "r_cis_responsive_only"])
    summary = {}
    if len(folds_df):
        summary["mean_r_trans_all_genes"] = float(folds_df.r_trans_all_genes.mean())
        summary["mean_r_cis_all_genes"] = float(folds_df.r_cis_all_genes.mean())
        w_all, p_all = stats.wilcoxon(folds_df.trans_minus_cis_all) if len(folds_df) >= 6 else (np.nan, np.nan)
        summary["wilcoxon_p_all_genes"] = float(p_all) if np.isfinite(p_all) else None
        summary["n_folds_trans_beats_cis_all_genes"] = int((folds_df.trans_minus_cis_all > 0).sum())
        summary["n_folds_total"] = int(len(folds_df))
    if len(valid_resp):
        summary["mean_r_trans_responsive_only"] = float(valid_resp.r_trans_responsive_only.mean())
        summary["mean_r_cis_responsive_only"] = float(valid_resp.r_cis_responsive_only.mean())
        diffs_resp = valid_resp.r_trans_responsive_only - valid_resp.r_cis_responsive_only
        w_r, p_r = stats.wilcoxon(diffs_resp) if len(diffs_resp) >= 6 else (np.nan, np.nan)
        summary["wilcoxon_p_responsive_only"] = float(p_r) if np.isfinite(p_r) else None
        summary["n_folds_trans_beats_cis_responsive_only"] = int((diffs_resp > 0).sum())

    ca24_folds = folds_df[folds_df.held_out_condition == "24hCA"]
    summary["highlighted_fold_24hCA"] = ca24_folds.to_dict(orient="records")

    if summary.get("wilcoxon_p_responsive_only") is not None and summary["wilcoxon_p_responsive_only"] < 0.05 \
            and summary.get("mean_r_trans_responsive_only", 0) > summary.get("mean_r_cis_responsive_only", 1):
        bifurcation = ("SUCCESS -- the lightweight trans-network model significantly beats the "
                        "cis-sequence baseline at zero-shot condition generalization. Code above "
                        "the ceiling exists; proceed to formalizing the full compositional architecture.")
    elif summary.get("wilcoxon_p_responsive_only") is None:
        bifurcation = "INCONCLUSIVE -- insufficient folds/genes to run the paired significance test."
    else:
        bifurcation = ("NOT CLEARED -- the trans-network model does not significantly beat the "
                        "cis-sequence baseline on this lightweight design. Does not by itself kill "
                        "Move 1 (mean-pseudobulk + TF-mRNA-as-activity-proxy + CollecTRI/fallback "
                        "network is a genuinely lightweight POC, not the full architecture) but does "
                        "not provide the proof-of-concept green light either -- report honestly, "
                        "flag for Manager/Brian, do not force a positive reading.")

    report["poc_summary"] = summary
    report["bifurcation_readout"] = bifurcation
    report["total_runtime_s"] = time.time() - t0
    with open(WORK / "scbloodnl_move1_poc_report.json", "w") as f:
        json.dump(report, f, indent=2, default=str)

    print(f"\n[BIFURCATION READOUT] {bifurcation}")
    print(f"\n[done] total runtime {report['total_runtime_s']:.0f}s")
    print(json.dumps(summary, indent=2, default=str))


if __name__ == "__main__":
    main()